# 05 — Volatility Engine

ATR, realized volatility, EWMA volatility and volatility percentile.


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Volatility Modules
from src.data.loaders import read_parquet, save_parquet
from src.features.returns import add_returns
from src.features.volatility import (
    add_atr,
    add_realized_volatility,
    add_ewma_volatility,
    add_volatility_percentile,
    classify_volatility_regime,
)
from src.features.trend import add_moving_averages
from src.features.distribution import add_rolling_distribution_stats
from src.features.liquidity import add_liquidity_features


In [3]:
# Cell: Build Volatility Features
clean = read_parquet("data/processed/clean_daily_ohlcv.parquet")
features = add_returns(clean)
features = add_atr(features, window=14)
features = add_realized_volatility(features, window=20)
features = add_ewma_volatility(features, span=20)
features = add_volatility_percentile(features, vol_col="realized_vol_20d", window=252)
features = classify_volatility_regime(features)
features.shape


(717835, 25)

In [4]:
# Cell: Inspect Volatility Outputs
features[["symbol", "date", "close", "atr_pct_14", "realized_vol_20d", "volatility_percentile", "volatility_regime"]].sort_values(["symbol", "date"]).groupby("symbol").tail(3)


,symbol,date,close,atr_pct_14,realized_vol_20d,volatility_percentile,volatility_regime
647,A32,2026-07-14,31000,0.037558,0.820379,0.454365,Normal Vol
648,A32,2026-07-15,29605,0.047622,0.834105,0.468254,Normal Vol
649,A32,2026-07-20,31000,0.041551,0.852176,0.484127,Normal Vol
2279,AAA,2026-07-16,7200,0.020933,0.164799,0.134921,Low Vol
2280,AAA,2026-07-17,7220,0.020182,0.162835,0.134921,Low Vol
...,...,...,...,...,...,...,...
716203,YBM,2026-07-15,9990,0.017518,0.236878,0.246032,Normal Vol
716204,YBM,2026-07-20,9660,0.020482,0.265111,0.297619,Normal Vol
717832,YEG,2026-07-16,8080,0.039869,0.422874,0.742063,High Vol
717833,YEG,2026-07-17,7950,0.035669,0.424575,0.746032,High Vol


In [5]:
# Cell: Save Volatility Features & Full Stock Features
# GUARDED WRITE (docs/reports/codebase_audit/MARKET_REGIME_AUDIT.md NV1): this
# path is the shared production stock_features.parquet, canonically produced
# by `python scripts/generate_features.py` from full_history_v2 with the full
# feature set (49 cols). This notebook only builds a subset of those columns
# -- running it by hand used to silently overwrite the production file with
# an incomplete version. Default: skip the write, just show the regime
# breakdown. Set ALLOW_PRODUCTION_WRITE=1 before starting the kernel if you
# really mean to overwrite it by hand.
from src.data.production_guard import guard_production_write, ProductionWriteBlocked

features = add_moving_averages(features, windows=[20, 50, 200])
features = add_rolling_distribution_stats(features, window=60)
features = add_liquidity_features(features)

output_path = "data/processed/stock_features.parquet"
try:
    guard_production_write(output_path)
    save_parquet(features, output_path)
    print(f"Da ghi {output_path} (ALLOW_PRODUCTION_WRITE=1).")
except ProductionWriteBlocked as e:
    print(f"KHONG ghi {output_path} -- {e}")

features["volatility_regime"].value_counts(dropna=False)


KHONG ghi data/processed/stock_features.parquet -- Refusing to write data/processed/stock_features.parquet -- this is a shared production file read by ~20 other scripts/modules (build_portfolio_stack.py, run_alpha_scoring.py, final_watchlist.py, ...). The canonical way to regenerate it is `python scripts/generate_features.py`, not this notebook. If you really mean to overwrite it by hand (e.g. debugging), set ALLOW_PRODUCTION_WRITE=1 before starting the kernel, or pass allow=True to guard_production_write(). See docs/reports/codebase_audit/MARKET_REGIME_AUDIT.md for the incident this guards against.


volatility_regime
Normal Vol    163390
Low Vol       139026
Unknown       137654
High Vol      132755
Stress Vol     79840
Panic Vol      65170
Name: count, dtype: int64